# Lab 00 — Deploy Infrastructure
### Fix It Fast: Autonomous Bug Resolution with Claude Code on Amazon Bedrock AgentCore

This notebook deploys all the infrastructure needed for the workshop. Run it once before participants open `lab-01-bug-fix-agent.ipynb`.

**What gets deployed:**

| Step | Resource | Time |
|---|---|---|
| 1 | GitHub repo + 9 bug issues | ~1 min |
| 2 | GitHub MCP Gateway (Secrets Manager + ECR + AgentCore Runtime + Gateway) | ~5–8 min |
| 3 | Shared infrastructure (CloudFormation: VPC + S3 Files) | ~3–5 min |
| 4 | Claude Code container (Docker build + ECR push) | ~3–6 min |
| 5 | AgentCore Runtime | ~2–3 min |

**Total: ~15–25 minutes**

---
### Prerequisites

| Requirement | Check |
|---|---|
| AWS CLI v2, valid credentials | `aws sts get-caller-identity` |
| Docker running, `buildx` available | `docker buildx version` |
| Python 3.10+ | `python3 --version` |
| `jq` installed | `jq --version` |
| `gh` CLI authenticated | `gh auth status` |
| GitHub App created (see Step 0 below) | — |

## Step 0 — Create a GitHub App (one-time)

Skip if you already have one.

**Create the app:**
1. Go to **https://github.com/settings/apps** → **New GitHub App**
2. Name: anything (e.g. `AgentCore Workshop`)  
   Homepage URL: `http://localhost`  
   Webhook: uncheck **Active**
3. Repository permissions: **Contents** Read & Write · **Issues** Read & Write · **Pull requests** Read & Write · **Metadata** Read-only
4. Click **Create GitHub App** → note the **App ID** at the top of the settings page
5. Scroll down → **Generate a private key** → a `.pem` file downloads automatically

**Install the app and find the Installation ID:**

6. In the left sidebar click **Install App** → click **Install** next to your account
7. After installing, look at the browser URL — it will look like:  
   `https://github.com/settings/installations/12345678`  
   The number at the end (`12345678`) is your **Installation ID**

> **Alternative:** run `gh api /app/installations` (requires a JWT) or check  
> `https://github.com/settings/installations` and click your app — the ID is in the URL.

## Step 1 — Configuration

In [ ]:
import os, subprocess, json, time, sys, shutil, tempfile, glob, re

# ── Fill these in ─────────────────────────────────────────────────────────────
GITHUB_USER        = "your-github-username"
GITHUB_REPO        = "my-task-manager"

GITHUB_APP_ID      = "your-app-id"          # e.g. "1234567"
GITHUB_APP_INSTALL = "your-installation-id"  # e.g. "12345678"

AWS_REGION         = "us-west-2"
AWS_PROFILE        = "default"
# ─────────────────────────────────────────────────────────────────────────────

WORKSHOP_DIR = os.path.abspath("")

_pem_files = glob.glob(os.path.join(WORKSHOP_DIR, "*.pem"))
GITHUB_APP_PEM = _pem_files[0] if _pem_files else "/path/to/your-app.private-key.pem"
print(f"  PEM file: {GITHUB_APP_PEM}")
assert os.path.exists(GITHUB_APP_PEM), f"PEM file not found: {GITHUB_APP_PEM}\nCopy your .pem into: {WORKSHOP_DIR}"

os.environ["AWS_REGION"]                   = AWS_REGION
os.environ["AWS_DEFAULT_REGION"]           = AWS_REGION
os.environ["AWS_PROFILE"]                  = AWS_PROFILE
os.environ["NO_COLOR"]                     = "1"
os.environ["GITHUB_APP_ID"]               = GITHUB_APP_ID
os.environ["GITHUB_APP_PRIVATE_KEY_FILE"] = GITHUB_APP_PEM
os.environ["GITHUB_APP_INSTALLATION_ID"]  = GITHUB_APP_INSTALL

GATEWAY_DIR     = os.path.join(WORKSHOP_DIR, "gateway_mcp")
INFRA_DIR       = os.path.join(WORKSHOP_DIR, "infra")
CLAUDE_CODE_DIR = os.path.join(WORKSHOP_DIR, "coding_agents", "claude-code")
SAMPLE_PROJECT  = os.path.join(WORKSHOP_DIR, "sample-project")

_ANSI = re.compile(r"\x1b\[[0-9;]*m")

def run(cmd, cwd=None, capture=False):
    env = os.environ.copy()
    if capture:
        r = subprocess.run(cmd, shell=True, cwd=cwd, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        r.check_returncode()
        return _ANSI.sub("", r.stdout).strip()
    subprocess.run(cmd, shell=True, cwd=cwd, env=env, check=True)

def read_dotconfig(path):
    cfg = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line and "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                cfg[k] = v.strip('"\'')
    return cfg

print("Configuration loaded.")
print(f"  Region:  {AWS_REGION}")
print(f"  Profile: {AWS_PROFILE}")
print(f"  GitHub:  {GITHUB_USER}/{GITHUB_REPO}")
print(f"  App ID:  {GITHUB_APP_ID}")

## Step 2 — Verify AWS credentials

In [ ]:
identity = json.loads(run("aws sts get-caller-identity --output json", capture=True))
print(f"Account: {identity['Account']}")
print(f"ARN:     {identity['Arn']}")

## Step 3 — Install Python dependencies

In [ ]:
run(f"{sys.executable} -m pip install -q -r {os.path.join(WORKSHOP_DIR, 'coding_agents', 'requirements.txt')}")
print("Dependencies installed.")

## Step 4 — Create the sample repository and seed issues

In [ ]:
result = subprocess.run(
    f"gh repo view {GITHUB_USER}/{GITHUB_REPO} --json name",
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(f"Creating repo {GITHUB_USER}/{GITHUB_REPO}...")
    with tempfile.TemporaryDirectory() as tmpdir:
        repo_path = os.path.join(tmpdir, GITHUB_REPO)
        run(f"gh repo create {GITHUB_USER}/{GITHUB_REPO} --private --clone", cwd=tmpdir)
        shutil.copytree(SAMPLE_PROJECT, repo_path, dirs_exist_ok=True)
        run("git add . && git commit -m 'Initial commit' && git push -u origin main", cwd=repo_path)
    print("Repo created and code pushed.")
else:
    print(f"Repo already exists, skipping create.")

print("\nSeeding issues...")
run(f"./seed-issues.sh {GITHUB_USER} {GITHUB_REPO}", cwd=WORKSHOP_DIR)
print(f"\nDone: https://github.com/{GITHUB_USER}/{GITHUB_REPO}/issues")

## Step 5 — Deploy the GitHub MCP Gateway

Deploys: GitHub App credentials → Secrets Manager · MCP server container → ECR · AgentCore Runtime (MCP) · AgentCore Gateway (IAM-auth)

**~5–8 minutes**

In [ ]:
# Ensure uv.lock is present and up to date (required by the gateway Dockerfile)
import shutil
assert shutil.which("uv"), "uv not found — install with: brew install uv"
run("uv lock", cwd=os.path.join(GATEWAY_DIR, "app"))
print("uv.lock ready.")

In [ ]:
STATE_FILE = os.path.join(GATEWAY_DIR, ".deployed-state.json")

GATEWAY_URL = ""
if os.path.exists(STATE_FILE):
    GATEWAY_URL = json.load(open(STATE_FILE)).get("gateway_url", "")

if GATEWAY_URL:
    print(f"Gateway already deployed: {GATEWAY_URL}")
else:
    print("Deploying GitHub MCP Gateway...")
    result = subprocess.run("./deploy-all.sh", shell=True, cwd=GATEWAY_DIR, env=os.environ.copy())
    if result.returncode != 0:
        raise RuntimeError(f"deploy-all.sh failed with exit code {result.returncode} — scroll up for the error output")
    GATEWAY_URL = json.load(open(STATE_FILE))["gateway_url"]
    print(f"\nGateway URL: {GATEWAY_URL}")

os.environ["GATEWAY_URL"] = GATEWAY_URL
print(f"GATEWAY_URL: {GATEWAY_URL}")

## Step 6 — Deploy shared infrastructure (VPC + S3 Files)

**~3–5 minutes**

In [ ]:
INFRA_CONFIG = os.path.join(WORKSHOP_DIR, "infra.config")

if os.path.exists(INFRA_CONFIG):
    print("Infrastructure already deployed.")
else:
    print(f"Deploying shared infrastructure in {AWS_REGION}...")
    run(f"./setup.sh {AWS_REGION}", cwd=INFRA_DIR)

cfg = read_dotconfig(INFRA_CONFIG)
print(f"VPC:      {cfg.get('INFRA_VPC_ID')}")
print(f"Subnet 1: {cfg.get('INFRA_SUBNET_1')}")
print(f"Subnet 2: {cfg.get('INFRA_SUBNET_2')}")
print(f"S3 Files: {cfg.get('INFRA_S3FILES_AP_ARN')}")

## Step 7 — Build and push the Claude Code container

**~3–6 minutes**

In [ ]:
AGENT_CONFIG = os.path.join(CLAUDE_CODE_DIR, "agent.config")

if os.path.exists(AGENT_CONFIG):
    ECR_URI = read_dotconfig(AGENT_CONFIG).get("ECR_URI", "")
else:
    ECR_URI = ""

if ECR_URI:
    print(f"Container already built: {ECR_URI}")
else:
    print("Building and pushing Claude Code container...")
    run("./setup.sh", cwd=CLAUDE_CODE_DIR)
    ECR_URI = read_dotconfig(AGENT_CONFIG)["ECR_URI"]

os.environ["ECR_URI"] = ECR_URI
print(f"Image: {ECR_URI}")

## Step 8 — Create the AgentCore Runtime

**~2–3 minutes**

In [ ]:
RUNTIME_CONFIG = os.path.join(CLAUDE_CODE_DIR, "runtime_config.json")

if os.path.exists(RUNTIME_CONFIG):
    RUNTIME_ARN = json.load(open(RUNTIME_CONFIG)).get("runtime_arn", "")
else:
    RUNTIME_ARN = ""

if RUNTIME_ARN:
    print(f"Runtime already deployed: {RUNTIME_ARN}")
else:
    print("Creating AgentCore Runtime...")
    run(f"{sys.executable} deploy.py", cwd=CLAUDE_CODE_DIR)
    RUNTIME_ARN = json.load(open(RUNTIME_CONFIG))["runtime_arn"]

print(f"Runtime ARN: {RUNTIME_ARN}")

## Step 9 — Deployment summary

Share the values below with participants (or pre-fill `lab-01-bug-fix-agent.ipynb`).

In [ ]:
config_out = {
    "GITHUB_USER": GITHUB_USER,
    "GITHUB_REPO": GITHUB_REPO,
    "AWS_REGION":  AWS_REGION,
    "AWS_PROFILE": AWS_PROFILE,
    "RUNTIME_ARN": RUNTIME_ARN,
    "GATEWAY_URL": GATEWAY_URL,
}
config_path = os.path.join(WORKSHOP_DIR, "workshop_config.json")
with open(config_path, "w") as f:
    json.dump(config_out, f, indent=2)

print("=" * 60)
print("Deployment complete!")
print("=" * 60)
for k, v in config_out.items():
    print(f"  {k}: {v}")
print()
print(f"Config saved → workshop_config.json")
print("Participants open: lab-01-triage-agent.ipynb")

## Cleanup

**Reset repo between runs** (close PRs, delete fix/* branches, re-seed issues):

```bash
./cleanup-repo.sh <your-github-username> my-task-manager
```

Or run from the notebook:

In [ ]:
# ── Reset repo (between workshop runs) ───────────────────────────────────────
# run(f"./cleanup-repo.sh {GITHUB_USER} {GITHUB_REPO}", cwd=WORKSHOP_DIR)

# ── Tear down all AWS infrastructure ─────────────────────────────────────────
# run(f"{sys.executable} cleanup.py", cwd=CLAUDE_CODE_DIR)
# run("./cleanup.sh", cwd=INFRA_DIR)
# run("./delete-all.sh", cwd=GATEWAY_DIR)

print("Cleanup commands are commented out for safety. Uncomment and run as needed.")